# Pixasonics: An Image Sonification Toolbox for Python

# Introduction

Pixasonics is a library for interactive audiovisual image analysis and exploration, through image sonification. That is, it is using real-time audio and visualization to listen to image data: to map between image features and acoustic parameters. This can be handy when you need to work with a large number of images, image stacks, or hyperspectral images (involving many color channels) where visualization becomes limiting, challenging, and potentially overwhelming.

With pixasonics, you can launch a little web application (running in a Jupyter notebook), where you can load images, probe their data with various feature extraction methods, and map the extracted features to parameters of synths, devices that make sound. You can do all this in real-time, using a visual interface, you can remote-control the interface programmatically, record sound real-time, or non-real-time with a custom script.

You can also run multiple instances of the app, or use it "headless mode" (without its visual interface), perhaps in a command-line script.

## If you are in a hurry...

In [ ]:
# ;pip install pixasonics

# quick workflow with a simple example
from pixasonics.core import App, Mapper
from pixasonics.features import MeanChannelValue
from pixasonics.synths import Theremin

# create a new app
app = App() # by default 500x500 pixels

# load an image from file
app.load_image_file("images/test.jpg")

# create a Feature that will report the mean value of the red channel
mean_red = MeanChannelValue(filter_channels=0, name="MeanRed")
# attach the feature to the app
app.attach(mean_red) # this adds it to the pipeline, so that it is updated every frame (the Probe moves)

# create a Theremin, a simple sine wave synth that we will use to sonify the mean pixel value
theremin = Theremin(name="MySine")
# attach the Theremin to the app
app.attach(theremin) # this adds it to the pipeline, so that its audio output is patched into the global audio graph

# create a Mapper that will map the mean red pixel value (within the Probe) to the frequency of the Theremin
red2freq = Mapper(mean_red, theremin["frequency"], exponent=2, name="Red2Freq") # cubic mapping curve for a more "linear" feel of frequency changes
# attach the Mapper to the app
app.attach(red2freq) # this adds it to the pipeline, so that it is updated every frame (the Probe moves)

# Toolbox Structure

Pixasonics is mainly designed to run in a Jupyter Notebook environment. (It does also work in command line scripts.)

At the center of pixasonics is the `App` class. This represents a template pipeline where all your image data, feature extractors, synths and mappers will live. The App also comes with a graphical user interface (UI). You can do a lot with a single `App` instance, but nothing stops you from spawning different `App`s with bespoke setups.

When you have your app, you load an image (either from a file, or from a numpy array) which will be displayed in the `App` canvas. Note that _currently_ your image data height and width dimensions (the first two) will be downsampled to the `App`'s `image_size` creation argument, which is a tuple of `(500, 500)` pixels by default. (This will be improved later, stay tuned!)

Then you can explore the image data with a Probe (represented by the yellow rectangle on the canvas) using your mouse or trackpad. The Probe is your "stethoscope" on the image, and more technically, it is the sub-matrix of the Probe that is passed to all `Feature` objects in the pipeline.

Speaking of which, you can extract visual features using the `Feature` base class, or any of its convenience abstractions (e.g. `MeanChannelValue`). All basic statistical reductions are supported, such as `"mean"`, `"median"`, `"min"`, `"max"`, `"sum"`, `"std"` (standard deviation) and `"var"` (variance), but you can also make your own custom feature extractors by inheriting from the `Feature` base class (stay tuned for a K-means clustering example in the Advanced Use Cases section!). `Feature` objects also come with a UI that shows their current values and global/running min and max. There can be any number of different `Feature`s attached to the app, and all of them will get the same Probe matrix as input.

Image features are to be mapped to synthesis parameters, that is, to the settings of sound-making gadgets. (This technique is called "Parameter Mapping Sonification" in the literature.) All `Synth`s (and audio) in pixasonics are based on the fantastic [SignalFlow](https://signalflow.dev/) library. For now, there are 5 `Synth` classes that you can use (unless you create your own: see the "Custom synths" section below): `Theremin`, `Oscillator`, `FilteredNoise`, and `SimpleFM`. Each `Synth` comes with a UI, where you can tweak the parameters (or see them being modulated by `Mapper`s) in real-time.

What connects the output of a `Feature` and the input parameter of a Synth is a `Mapper` object. There can be multiple `Mapper`s reading from the same `Feature` buffer and a `Synth` can have multiple `Mapper`s modulating its different parameters.

# The App

The `App` class is at the core of the pixasonics workflow. The `App` is where you load your image data, where you move the probe, the `App` connects to the real-time audio server, and it represents the pipeline of `Feature`s connected to `Synth`s with `Mapper`s.

To showcase the `App` and its functionality, let's create a basic scene, using an image from the [CELLULAR open dataset](https://zenodo.org/records/8315423) and map the mean red channel value to the frequency of a Theremin (a simple sine wave generator).

In [ ]:
from pixasonics.core import App, Mapper
from pixasonics.features import MeanChannelValue
from pixasonics.synths import Theremin

# create a new app
app = App() # by default 500x500 pixels

# load an image from file
app.load_image_file("images/cellular_dataset/merged_8bit/Timepoint_001_220518-ST_C03_s3.jpg")

# create a Feature that will report the mean value of the red channel
mean_red = MeanChannelValue(filter_channels=0, name="MeanRed")
# attach the feature to the app
app.attach(mean_red) # this adds it to the pipeline, so that it is updated every frame (as the Probe moves)

# create a Theremin, a simple sine wave synth that we will use to sonify the mean pixel value
theremin = Theremin()
# attach the Theremin to the app
app.attach(theremin) # this adds it to the pipeline, so that its audio output is patched into the global audio graph

# create a Mapper that will map the mean red pixel value (within the Probe) to the frequency of the Theremin
red2freq = Mapper(mean_red, theremin["frequency"], exponent=2, name="Red2Freq") # cubic mapping curve for a more "linear" feel of frequency changes
# attach the Mapper to the app
app.attach(red2freq) # this adds it to the pipeline, so that it is updated every frame (as the Probe moves)

A few things just happened. The app created (or connected to) the global audio graph and the UI popped up: a canvas with the image on the left, and various settings panes on the right. The size of the canvas is determined by the `image_size` argument at the creation of the `App` object and cannot be changed afterwards. It is a tuple corresponding to `(height, width)`, and it is `(500, 500)` by default. All images that you load into the app will be __resized__ to this height and width! (This might be changed/improved in the future.) This means, that, at least currently, the app does not respect the aspect ratio of your input image, it will stretch or shrink it to whatever `image_size` your `App` has. Luckily the image we loaded is 2048x2048 with a square aspect ratio, so the default `image_size` was fine.

Now let's look to the right where the various settings panes live. These can be opened and closed, one at a time. Here is a brief summary of what you can find in them.

## Audio Settings

Click on "Audio Settings" on the top right to open this pane. Here you can control the global audio settings of the app.

### The Audio switch

There is an audio switch at the top left, and a volume slider next to it. To have any sound produced by the app, you need to turn on audio, either on the UI, or by evaluating:

In [ ]:
app.audio = True

Still no sound from the app, huh? This is because you (probably) haven't interacted with the canvas yet. Try clicking on a few of those _Drosophila_ S2 blood cells. Then try to click and drag to hear a continuous change in frequency.

### The Master Volume Slider

Next to the audio switch there is the Master Volume slider where you can set the loudness of the app's sound output in decibels (dB). You can either control it via the UI or the `master_volume` property:

In [ ]:
app.master_volume = -24 # check the slider after evaluating this cell

### Real-time recording to file

Below the audio switch and the master slider, there is a pair of widgets that let you record the real-time output of the app. Leave the file name at the default `"recording.wav"`, hit the Record button, click and drag around the image, then click on the Record button again to stop recording.

Let's listen to what you've done:

In [ ]:
from IPython.display import Audio, display
display(Audio("recording.wav"))

The global recording is meant to be a quick-and-easy way to record your results. If you want to be more precise (or faster than real-time), there are methods to render your results in non-real-time, which we will discuss later.

### The Master Envelope

Below the recording section in the Audio Settings pane you find the Master Envelope, which controls how fast the sound fades in and out when you interact with the canvas. It is a traditional Attack-Decay-Sustain-Release (ADSR) curve that is applied to the volume of the global audio output. Here is how it works in a nutshell:
- Attack: the time for the sound to fade in (in seconds),
- Decay: the time to fade to the Sustain level (in seconds),
- Sustain: the level (amplitude, between 0 and 1) to sustain indefinitely, until we release the mouse button (or deactivate the Probe),
- Release: the time for the sound to fade out after we release the mouse button (or deactivate the Probe).

There are some number fields to set these parameters, and a little drawing that visualizes the proportions of the different time segments. On the bottom, there is also a "Duration" value that shows the total duration of the envelope (that is Attack time + Decay time + Release time).

Let's set a bit slower envelope to illustrate how it works:

In [ ]:
app.master_envelope.attack = 0.5
app.master_envelope.release = 1.5

We can also set a more percussive envelope:

In [ ]:
app.master_envelope.attack = 0.01
app.master_envelope.decay = 0.01
app.master_envelope.sustain = 0.1
app.master_envelope.release = 0.8

For now, let's reset it to the default:

In [ ]:
app.master_envelope.attack = 0.1
app.master_envelope.decay = 0.01
app.master_envelope.sustain = 1
app.master_envelope.release = 0.1

While in the future Envelopes will have more functionality in Pixasonics, for now there is only a single global Master Envelope.

### Audio IO Settings

A quick technical detail while we are discussing Audio Settings. In an `App`, you can only query the sample rate or the output buffer size with the corresponding properties:

In [ ]:
app.sample_rate, app.output_buffer_size

However, these are actually settings for the global audio graph that is shared between all `App`s. It is a `signalflow.AudioGraph` object:

In [ ]:
app.graph

I you want to *change* the settings such as sample rate, buffer size, or the device being used, you have to change it globally, for all `App`s. You can do this via the `AudioIOSettings` *singleton*. (Singleton means, that there is only one instance of this object.) As with most classes in Pixasonics, it comes with its own UI which you can view by evaluating its `ui` property:

In [ ]:
from pixasonics.core import AudioIOSettings
audio_io_settings = AudioIOSettings()
audio_io_settings.ui

Due to some current limitations in SignalFlow, you cannot see the current state of the audio graph the moment you create this widget, which is indicated by the "Create Graph" button having a neutral gray color. Once you click on it, a new `signalflow.AudioGraph` is created (and then all `Synth`s and `App`s getting migrated to it). Upon success, the button should turn green. Try to create a new audio graph with a sample rate of 48000 Hz and a buffer size of 256 samples.

Now let's change the buffer size to 128 samples using the dropdown menu on the UI. Notice that once you changed a setting, the "Create Graph" button turns orange, indicating that there are changed settings that haven't been yet used to create a new graph. Click on the button again to apply the settings.

You can also view and change the settings using corresponding properties and methods:

In [ ]:
# view the current audio settings
audio_io_settings.backend, audio_io_settings.input_device, audio_io_settings.output_device, audio_io_settings.sample_rate, audio_io_settings.buffer_size

In [ ]:
# change the sample rate to 44100 Hz
audio_io_settings.sample_rate = 44100

# change the output buffer size to 512 samples
audio_io_settings.buffer_size = 512

Notice that after evaluating the cell above, the "Create Graph" button again turned orange. Let's apply the new settings through code:

In [ ]:
audio_io_settings.create_audio_graph()

Note that the button on the widget above turned green again. Let's now change back the sample rate to 48000 Hz, and the buffer size to 128 samples:

In [ ]:
audio_io_settings.sample_rate = 48000
audio_io_settings.buffer_size = 128
audio_io_settings.create_audio_graph()

Finally, if the set of audio devices connected to your computer changes (after you plugged in your headphones or a USB audio interface), you can refresh the list of devices available to Pixasonics (and SignalFlow) by clicking on the "Refresh Devices" button or evaluating:

In [ ]:
audio_io_settings.refresh_devices()

You can then change the output device via the widget or by setting the value of the corresponding property:

In [ ]:
audio_io_settings.output_device = "External Headphones" # change this to the name of your output device
audio_io_settings.create_audio_graph() # this will recreate the audio graph with the new output device

Now let's continue exploring our `App` and its Display Settings.

## Display Settings

By the way, we have started to stray a bit far from the cell where our app UI lives... Luckily, we can get a synced copy down here if we evaluate:

In [ ]:
app.ui

Aaah, much better. Let's open the Display Settings. As the name suggests, this is where you can find settings related to how the image is displayed in the app canvas. These will __only affect the display__ though, the underlying image data (that may be HDR, or have many color channels or image layers) will not be affected.

### Normalization

On the top of the pane you can find two checkboxes: "Normalize display" and "Global normalization". More often than not we need to normalize images to understand (or even, simply, to see!) the image content better. Traditionally, the normalization is performed individually on all color channels (Red, Green, and Blue in "normal" images). Keeping it channel-wise can help remove imbalances between the channels, like the green tint here. Check "Normalize display" in the UI, or evaluate:

In [ ]:
app.normalize_display = True

You probably noticed that the background became much less green-tinted. Let's try also checking "Global normalization":

In [ ]:
app.normalize_display_global = True

...and now the gren tint is back. It makes sense, because when normalization is set to global, the algorithm will scale all pixels according to the minimum and maximum pixel value apparent in __any__ of the channels. Since in the original image the green tint was the result of generally much higher green pixel values, applying global normalization brought the tint back. Let's turn it off for now:

In [ ]:
app.normalize_display_global = False

#### Normalization via percentiles

Sometimes there may be extremely bright or dark regions in your image. This means that even if you normalize it, you may not see the contents properly. In these cases, you can use the "Low/High Percentile" sliders under the normalization checkboxes, or define the percentiles programmatically:

In [ ]:
app.display_normalization_low_percentile = 5
app.display_normalization_high_percentile = 95

Notice how the image now has a strong blue tint. We can correct this by assigning different percentiles to the different color channels:

In [ ]:
# this will normalize the red and green channels to the 5th and 95th percentiles, but the blue channel to the 5th and 100th percentiles
app.display_normalization_high_percentile = [95, 95, 100]

Now the blue tint is gone, together with the slider, since we use different values for different channels. We can bring back the slider by assigning the same value to all channels again:

In [ ]:
app.display_normalization_high_percentile = 100

### Maximum displayed channels, Channel Offset, and Layer Offset

In the bottom section of the Display Settings pane, we have a number box for the *Maximum displayed channels* (currently set to 3), a slider for *Channel Offset*, and a (currently disabled) slider for *Layer Offset*. Let's break down what these do.

Maximum diplayed channels controls up to how many consecutive color channels are displayed. Since on a screen we can only show up to 3 color channels (corresponding to Red, Green, and Blue or: RGB), the maximum (and default) value of this property is set to 3:

In [ ]:
app.maximum_displayed_channels

However, it does not necessarily mean that we *always* see 3 channels. Try to move the Channel Offset slider one step or evaluate:

In [ ]:
app.display_channel_offset = 1

Since we offset the displayed color channels by one, now we see only two channels:
- the 2nd channel (Green) as the 1st *displayed* channel (Red), 
- the 3rd channel (Blue) as the 2nd displayed channel (Green),
- while the 1st channel is not displayed,
- and the 3rd displayed channel being empty (i.e., there is no amount of blue in the displayed image).

This may not make too much practical sense now, but it could be useful when viewing images with lots of color channels (such as hyperspectral images), where the combined use of `maximum_displayed_channels` and the `display_channel_offset` can help visually exploring the image volume.

For now, let's view the color channels one by one via:

In [ ]:
app.maximum_displayed_channels = 1

Now drag around the Channel Offset slider to see the three channels of the loaded image individually as grayscale displayed images. Note that we haven't altered the underlying image data (that we would sonify), we only changed the display (what we "draw" in this window).

As you have probably noticed, the Layer Offset slider is currently disabled. We will use it when we read NumPy arrays instead of single image files (stay tuned!). For now, let's recall the default settings:

In [ ]:
app.maximum_displayed_channels = 3
app.display_channel_offset = 0

...and move on to the Probe Settings.

## Probe Settings

First, let's create another view of our `App` to avoid scrolling (too much):

In [ ]:
app.ui

Close the Display Settings pane, and open the Probe Settings. This is where you set everything related to the Probe (remember, our image "stethoscope") and how you interact with it. By default, the Probe is represented by a rectangle drawn over the image. When the Probe is active, the rectangle is red, and when the Probe is inactive, the rectangle is yellow. (Also, when the Probe is active, the `App` will evaluate all attached mappings in the pipeline and unmute audio for all attached synths.)

There are many settings here, some of which you may not see. Try to hover over the Probe Settings pane and scroll with your mouse. Notice how some of the settings (related to masks) are disabled. We get back to these in the Advanced Use Cases section below.

### The Probe Width and Height

The width and height of the Probe can vary from 1x1 pixel up to the app's `image_size`. Try moving the sliders and observe how the shape of the yellow rectangle on the canvas changes. There is not really a "right" setting for this, it is highly dependent on the content of your image, the area of pixels you want to send to feature extraction, and generally, the kind of sonification you want to create. At an extreme, you can create horizontal or vertical scan lines for the whole image like this:

In [ ]:
# horizontal scanning (a vertical scan line)
app.probe_width = 1
app.probe_height = app.image_size[0] # remember that image size is given as (height, width)

In [ ]:
# vertical scanning (a horizontal scan line)
app.probe_width = app.image_size[1]
app.probe_height = 1

Most of the time, you probably want to set the Probe dimensions to something smaller to fit the content of the image. Let's set it to fit that cell in the middle:

In [ ]:
app.probe_width = 33
app.probe_height = 32
app.probe_x = 233
app.probe_y = 252

Yes, you guessed it: we can programmatically move the probe around using the `probe_x` and `probe_y` properties of our `App`. Setting the dimensions and the position of the probe like this will be the basis of scripting custom paths and render them non-real-time (more on that later).

### Interaction modes

Now something fun: interaction. So far we have used the "Hold" mode, that is, sound will be activated (and all existing mappers evaluated in the pipeline) while the mouse button is held down. This mode is comfortable when you want to quickly inspect (and listen to) the various parts of your image.

But sometimes you may want to activate the Probe, and then leave it on while you experiment with, let's say, changing synthesis parameters. Or you just want to free your mouse to use it somewhere else without stopping the sound output. This is what "Toggle" mode is for. You can change it on the UI by clicking on the "Toggle" button, or programmatically like so:

In [ ]:
app.interaction_mode = "toggle"

Now you can double-click on the probe to activate it. Double-click again to deactivate it. It could be sometimes comfortable to activate the Probe, and just click on various parts in the image (here, on different cells) to get an abrupt comparison of their sounding mappings. 

Try to activate the toggle and then click on the different cells to find out which one produces the highest pitch! Since we mapped the mean red channel value to the theremin frequency, and since in this image red fluorescent protein indicates the level of autophagy going on in the cell, we can intuitively find the cell with the most active autophagy by simple clicking around and listening. (Technically, if we would extract the _maximum_ red value under the probe we would get more precise results for this task, but let's give this a pass for now.)

The third interaction mode is called "Select", and it lets us setting the Probe width and height by drawing a bounding box over the image. Try it:

In [ ]:
app.interaction_mode = "select"

In select mode, we cannot activate (or deactivate) the Probe by mousing. So before moving on, let go back to Toggle mode:

In [ ]:
app.interaction_mode = "toggle"

The Select interaction mode will get another function when using instance masks (stay tuned for the examples below).

### Following the "idle" mouse

Below the interaction mode selector, there are some check boxes. The first one is for having the Probe __always__ follow your mouse, even when you don't hold the mouse button (i.e., when it's "idle"). Beyond this being simply comfortable for your hand if you are making sound from the image continuously, it can also be handy when you don't want sound output at the moment, just want to read the value of a Feature. Let's check "Probe follows idle mouse" on the UI, or evaluate:

In [ ]:
app.probe_follows_idle_mouse = True

Now, if you hover your mouse over the app canvas, you can see that the Probe constantly follows it, and you can see indicators of the Probe's horizontal and vertical coordinates (at the bottom of the pane) labelled "Probe X" and "Probe Y", respectively, and controlled by the `probe_x` and `probe_y` properties (as shown above). Before moving on to the filter check boxes, let's turn this option off:

In [ ]:
app.probe_follows_idle_mouse = False

### Filter the Probe by Channel or Layer offset

There is a bit technical discussion ahead, so feel free to skip to the next section. TLDR: the filtering check boxes are handy when dealing with image volumes that have many channels and layers, but we will do a quick example here that involves (temporarily) modifying our `mean_red` `Feature`.

Let's bring the `App` UI here:

In [ ]:
app.ui

Under the "Probe follows idle mouse" check box, there are two check boxes for filtering the Probe by the current (display) Channel offset or Layer offset. To understand what these options enable, let's recap on what the Probe does.

Normally, when *not* using instance masks, the Probe creates a rectangular slice from the loaded (not the displayed) image. As we will see in the Advanced Use Cases section, we can load more than a single RGB image; we can load images with more (or less) color channels, and we can also load several of such images onto *layers*, for example to sonify an image stack or a time lapse sequence.

In all cases, the Probe slices *in all dimensions* simultaneously. Which means, if we stay with the current sonification setup, unless we configure a `Feature` to only look at certain color channels or layers in the loaded image volume, the sonification will be based on *all* channels and layers within the Probe's slice. Sometimes, we may want to listen to the same sonification pipeline but only based on the currently viewed image slice, or, in other words: the image slice *filtered* by the current Channel or Layer offset. This way, we can step through the channels/layers, and hear the sonified result change accordingly. (Imagine stepping through a time lapse this way to listen to how the same area changes over time.)

While the probe-filtering options are typically useful when dealing with a large(r) image volume with many channels and layers, we can temporarily modify our `MeanChannelValue` `Feature` (saved in the `mean_red` variable) to illustrate what filtering the Probe achieves. This gets a bit into the underlying structure of the `Feature` class, but don't worry, there is a more in-depth explanation below (under "The Feature base class..." section). If the current example gets a bit too complex, feel free to skip this, you will understand more once we discuss `Feature`s in more detail.

When creating this sonification pipeline, we used a `MeanChannelValue` which is actually a shortcut for initializing a `Feature` that has its `reduce_method` property set to `"mean"`, its `filter_channels` set to `0` (meaning the 1st, Red, channel), and its `target_dim` (the dimension it should reduce to) set to `2`, which is the Channel dimension (from Height, Width, Channels, Layers). This means our `Feature`, when evaluated, filters the received Probe matrix to only process the Red channel, than takes the mean over all dimensions but the Channel dimension.

Because of this setup, even if we would enable "Filter probe by Channel Offset", our `mean_red` would still only look at the 1st, Red, channel. To make our example work, we have to switch of its filtering (setting `filter_channels` to `None`) *and* set its `target_dim` to `-1`, which means it will reduce the whole Probe matrix to a *single* value (here: by taking the mean of the matrix).

In [ ]:
mean_red.filter_channels = None
mean_red.target_dim = -1

Now, in the `App`, we can enable filtering the Probe by the channel offset. To get a visual understanding of the sounding result, let's also set the maximum displayed channels to 1.

In [ ]:
app.filter_probe_by_channel_offset = True
app.maximum_displayed_channels = 1

Now enable the Probe by double-clicking on it (remember, we are still in Toggle mode), then go to the Display Settings, move the Channel Offset slider back and forth and listen to the result. With this setup we can compare the intensity of the different channels (within the Probe) with the same sonification pipeline. As mentioned above, using the probe-filtering settings will come more handy when sonifying image volumes with many channels and layers (stay tuned for the advanced examples below!). For now, let's keep on plowing through the basics, and set our `MeanChannelValue` and `App` back to how they were:

In [ ]:
mean_red.filter_channels = 0 # only look at the red channel (channel 0) when computing the mean value
mean_red.target_dim = 2 # compute the mean value across the channels
app.filter_probe_by_channel_offset = False # do not filter the probe by channel offset
app.maximum_displayed_channels = 3 # display all channels in the probe

The rest of the controls in the Probe Settings will be available once we load an instance mask (for example to isolate a single cell in the image). We will discuss this scenario in the Advanced Use Cases section below. For now, let's move on to the Features pane.

# Features

The Features pane is where all your `Feature` objects (that you attached to the `App`!) will show up. (As of now, in Pixasonics, the term "feature" always refers to a _visual_ feature, as currently audio feature extraction and mapping to auditory features is not supported.) 

Right now you should see a card labelled "MeanRed" there, our Feature that reports the mean red pixel value of the Probe's slice of the image. Let's enable the option for the Probe to follow our idle (hovering) mouse, move the Probe around a bit (without necessarily making any sound), and look at the values we get on the card.

To avoid scrolling back-and-forth too much, let's have another copy of our app UI here:

In [ ]:
app.probe_follows_idle_mouse = True
app.ui

Every feature gets the slice of the image data under the Probe and reduces it into one or more values. Every time you move the probe with your mouse, all the features within the `App` are sent a new Probe matrix, and they update their feature buffers (the container where they store their most recent values). This example feature that we named "MeanRed" measures the mean value of the Probe matrix in the red color channel.

## The header

On the card you can see that, next to the name, there is a small identifier, which is uniquely generated for every different object. It is not very important now, but it might help you later identifying various objects in your pipeline.
Underneath this header you see the "Number of Features:" box that shows you how many values does this `Feature` report. Now it's just 1, since we are only filtering for the first (red) channel. Before we discuss what's behind this in more detail, let's just move on and look at the rest of the UI.

## The values

Most of the card is occupied by 3 number fields, showing the minimum, maximum and current (or most recent) value of the `Feature`. When a `Feature` is attached to the `App`, it actually gets to look at the whole image _once_, so it can calculate things like global minimum and maximum values. You probably noticed that as you move the Probe around, the Min and Max fields don't change. That is because it calculated the global minimum and maximum value in the entire image.

## Resetting to running min and max

Keeping track of the range of values will come especially handy when setting up mappings. In many cases it is helpful to get global min/max values, but sometimes we might need running min/max values of the Probe matrix only. If you hover your mouse over the Reset button in the bottom left corner of the Feature card, you will see the tooltip appear: _"Reset to the running min and max"_. You can click on that or evaluate:

In [ ]:
mean_red.reset_minmax()

Notice that at the time of resetting you got the same value as min and max. But when you move the Probe around, you see that the values start to move away from each other, as the running min and max values update. This behavior can be useful when you want to create a dynamic mapping and compare two visual objects relative to each other. It can also be convenient to compare parts of the image only visually, without sonification.

## Attach to and detach from the App

But what about the other button that says "Detach"? Click on it, or evaluate:

In [ ]:
app.detach(mean_red)

Oh no, our `Feature` is gone! Don't worry, the `mean_red` variable still holds our `Feature` object, it's just no longer included in the `App`'s pipeline. That is, whenever the Probe moves, this detached `Feature` won't get the updated Probe matrix anymore, and its values (its feature buffer) will remain the same. Just to prove that it's still there, here is how you can pull up its UI separately:

In [ ]:
mean_red.ui

There you go, our MeanRed feature is still functional. It is just outside the `App`'s pipeline. But we can attach it back by saying:

In [ ]:
app.attach(mean_red)

If you go to the `App` UI again, you can see that now the card labelled MeanRed re-appeared in the Features pane. And by the way, our copy of the feature UI still works, it is in sync with the one in the `App` UI. Try to move the Probe and you'll see the numbers update in both places.

This mechanism of "attaching" and "detaching" objects to and from the main app is a pattern that is the same for `Feature`s, `Synth`s, and `Mapper`s. When you want to include something in the `App`, which is to say, you want to include it in the pipeline of Probe --> Features --> Mappers --> Synths, you _attach_ it to the App. When you don't need it anymore (even just for a while), you can _detach_ it from the App. The object will still "live", but it won't get updates from the `App` anymore.

Now let's create another `Feature` to see the mean pixel values across all three color channels:

In [ ]:
mean_pix = MeanChannelValue(name="MeanPix")

Recall how we created `mean_red`? It was like this:
```python
# create a Feature that will report the mean value of the red channel
mean_red = MeanChannelValue(filter_channels=0, name="MeanRed")
```

Notice that the only difference is that this time we did not specify `filter_channels` (so it defaults to `None`). By default all `Feature` objects process the entire Probe matrix which is actually a 4-dimensional matrix in the shape of:

__(Height, Width, Channels, Layers)__

_In fact, our entire image data is kept in that shape._ So when we specified `filter_channels=0`, we selected the 0th (red) channel in the 2nd, Channel dimension. Don't worry if this is a bit over your head now, the TLDR is that `Feature` objects can actually filter the Probe matrix in any way you need.

Now let's attach our new `Feature` to our `App`:

In [ ]:
app.attach(mean_pix)

Notice that the Features pane on our app UI has two cards now: MeanRed and MeanPix. We can also query the currently attached `Feature`s like so:

In [ ]:
app.features

As you have probably noticed, MeanPix reports 3 values, which are the mean pixel values across the 3 channels of the Probe matrix. We'll get into more details about `Feature`s in a bit, but for now let's move on to `Synth`s.

# Synths

`Synth`s are the tools to make sound with in pixasonics. As mentioned at the top, they are all abstractions (technically, `Patch`es) based on the awesome [SignalFlow](https://signalflow.dev/) library.

First let's have a look at a simple Theremin synth, outside the `App`. Here is how you make one:

In [ ]:
simple_theremin = Theremin(name="SimpleTheremin")

We can pull up its UI like so:

In [ ]:
simple_theremin.ui

As you see we got a similar card here: a header with the name of the synth and its unique ID, a main section with all its parameters and corresponding sliders, as well as Reset and Detach buttons at the bottom of the card.

## The graph

There is nothing there though that turns the synth on or off. This is normally done by our `App` as a result of our interaction with the Probe. Under the hood, the `App` creates (or connects to) the global audio graph in SignalFlow, which is the `AudioGraph` object. This way we can use `App.graph` to reference this global audio graph and play the `output` Node of our synth patch:

In [ ]:
graph = app.graph
graph.play(simple_theremin.output)

## The Parameters and Resetting

Now you should hear a static sine tone. Try moving the sliders and listen to how the sound changes to get an understanding of the parameters. When you are done, you can hit the Reset button to reset all parameters to their defaults. These defaults are not set in stone, they are decided when we create the synth object. If we spell it out, we actually said this:
```python
simple_theremin = Theremin(frequency=440, amplitude=0.5, panning=0, name="SimpleTheremin")
```

You can also set parameters programmatically like so:

In [ ]:
simple_theremin.set_input("frequency", 880) # set frequency to 880 Hz
simple_theremin.set_input("amplitude", 0.2) #set amplitude to 0.2

Wonder what other parameters are available to set? You can always look at the `params` attribute which holds a dictionary:

In [ ]:
simple_theremin.params.keys()

Now let's stop playing the synth:

In [ ]:
graph.stop(simple_theremin.output)

## Mapping to parameters

In Pixasonics, the main sonification technique we use is Parameter Mapping Sonification. This is the most common sonification technique, where data features get mapped to input parameters of a synthesizer. While this might have some perceptual issues when mappings get complex, there is a reason why this technique is so popular: it is very simple to set up and it is easily explainable.
There is a way to define your custom mapping scheme too, stay tuned for that (or jump ahead to the "Advanced Use Cases" section).
For now, let's focus on how we can express quantities in the image (such as the level of red channel values) as changes in the parameters of synths, such as the frequency.

Let's make a new instance of our app UI again for convenience:

In [ ]:
app.ui

## Attach and detach

Open the Synths pane. Just like the Features pane, this is where all the attached `Synth`s will have their UIs. Similarly to `Feature`s, you can detach `Synth`s by clicking on the Detach button on their card, or by calling:

In [ ]:
app.detach(theremin)

This means our Theremin is not included in the `App`'s audio graph anymore, so when we activate the probe, there is no sound (since there is no `Synth` currently attached to the `App`).

We can attach the Theremin back by calling:

In [ ]:
app.attach(theremin)

## Moving sliders

Now try to activate the Probe, listen to the sound and watch the frequency slider of the Theremin. (Remember, we are still in "Toggle" interaction mode, so you have to double-click to activate the Probe.)

The slider of the Theremin seems to move on its own as we move the Probe. (Don't worry that the slider updates are not constant, this is a deliberate optimization to avoid choking the Jupyter server with slider updates.) Have you observed any relationship between the image data and the frequency of the Theremin? If you need a hint, open the Mappers pane on the app UI.

# Mappers

In our very first setup of this scene, we created and attached this `Mapper` like so:
```python
# create a Mapper that will map the mean red pixel value (within the Probe) to the frequency of the Theremin
red2freq = Mapper(mean_red, theremin["frequency"], exponent=2, name="Red2Freq") # cubic mapping curve for a more "linear" feel of frequency changes
# attach the Mapper to the app
app.attach(red2freq) # this adds it to the pipeline, so that it is updated every frame (as the Probe moves)
```

`Mapper`s take two positional arguments: the `Feature` object we want to map _from_ and the `Synth` parameter (or a list of `Synth` parameters!) we want to map _to_. The notation `theremin["frequency"]` will tell the `Mapper` to map to the frequency parameter of the theremin `Synth` object. The actual object that is returned is a dictionary:

In [ ]:
theremin["frequency"]

## Ranges at the input and output sides

There is a lot of info in here that we don't need to worry about right now. The most important for us now are the two top ones, the 'min' and 'max'.

What a `Mapper` does, is mapping a value (or an array of values) from a given input range to a given output range. It can do this either linearly (where `exponent=1`) or non-linearly (like `exponent=2` for a cubic curve).

But we haven't specified any input or output range when we created our `red2freq` mapper object. How can it still know what red values to map to what frequency values?

## Default ranges

The trick is knowing that, on one hand, `Feature` objects report their global (or running) min and max values (remember?!), and, similarly, that all `Synth` parameter dictionaries will contain curated (sensible) min and max values. If no input and output ranges are specified upon creation, the `Mapper` object will just look up the corresponding min and max values on both the feature-side and the synth-side.

## Specific ranges

Having automatic ranges that update live with the running min/max of a `Feature` object can be handy for experimentation, but sometimes we do want to specify something more concrete. Before we get there, it is perhaps helpful to look at the spelled-out verison of how we created our `red2freq` object:
```python
red2freq = Mapper(
    obj_in=mean_red, 
    obj_out=theremin["frequency"],
    in_low=None,
    in_high=None,
    out_low=None,
    out_high=None,
    exponent=2, 
    clamp=True,
    name="Red2Freq")
```

We can specify an input range (on the `Feature` side) with the `in_low` and `out_low` arguments, while the output range (on the `Synth` parameter side) with the `out_low` and `out_high` arguments. They all default to `None`, which means that the `Mapper` will look up the current min/max reported by the `Feature` and the default min and max values of the `Synth` param (that were decided when we created the `Synth`).

We can also retrieve these values like so:

In [ ]:
red2freq.in_low, red2freq.in_high, red2freq.out_low, red2freq.out_high

Note that even though our MeanRed `Feature` reports singular scalar values, the data is in a Numpy array of shape `(1, 1)`. This technical detail will be more important when we want to map a vector of features to a multichannel synth, but let's ignore that case for now.

What is important is that we can now override the ranges like so:

In [ ]:
red2freq.in_low = 150
red2freq.in_high = 200

Now let's listen to what has changed in the mapping.

You probably notice that for most cells the sound remains at its minimum pitch, while for the really bright ones it suddenly chirps up to a really high whistle. Let's go back to using the automatic ranges by setting the `in_low` and `in_high` props to `None`:

In [ ]:
red2freq.in_low = None
red2freq.in_high = None

Take a listen!

We are back to the kind of mapping we started out with. It is striking how little change resulted in such a drastic change in the sound pattern, isn't it?

## Clamp

And by the way, in the previous setting, why did all the cells that were not so bright get the same static low hum? That is because by default we clamp (limit) the mapped values to `out_low` and `out_high`. You probably want clamping to be on for most of the time, but if you know what you are doing, consider turning it off (by specifying `clamp=False`) and then any input value that is below `in_low` or above `in_high` will get proportionally mapped to below `out_low` or `out_high`. But be careful, this can quickly explode values, especially when you use a non-linear scaling.

## Exponent

The last important parameter of a `Mapper` is the exponent to use for the mapping. Think about it as the mapping "curve". To get a sense of how `exponent` influences the mapping curve, consider this interactive plot:

In [ ]:
from pixasonics.ui import ExponentPlot
exponent_plot = ExponentPlot()
exponent_plot()

For linear mapping we can use (the default) `exponent=1`:

In [ ]:
exponent_plot.exponent = 1

But we might want an cubic or a logarithmic curve for our mapping. In the cells below you can find some examples, but feel free to explore the curve by moving the Exponent slider!

In [ ]:
exponent_plot.exponent = 2

In [ ]:
exponent_plot.exponent = 4

In [ ]:
exponent_plot.exponent = 0.5

In [ ]:
exponent_plot.exponent = 0.25

Since we were mapping to frequencies (measured in Hz) we actually get preceptually more "linear" feel in the mapping if the curve is cubic, that is why we set `exponent=2` in our `red2freq` object.

In [ ]:
red2freq.exponent

# Advanced Use Cases

So far we have covered the basics of Pixasonics, how you can set up an `App` with a pipeline that involves various `Feature`s, `Synth`s and `Mapper`s. Additionally, there are a few "breakout" doors designed to integrate Pixasonics in your existing workflow or to speed/scale up your sonification sessions:
- __Loading NumPy arrays__: An `App` lets you load any matrix data (up to 4 dimensions) as an image to sonify. If you have any specific preprocessing, you can set it up to output Numpy matrices which you can then load into the `App`. Using NumPy arrays also lets you load image sequences or hyperspectral images (there is no conceptual restriction of the number of color channels or image layers used). By the way, if you don't want to worry about NumPy arrays, you can also directly load HDR images from files.
- __Using instance masks for probing__: You are not restricted to Probe the loaded image via rectangular slices. Pixasonics supports instance masks: image tensors, where pixel values represent object instances. This way you can make sure that the sonification is only based on the pixels of an object (for example the pixels that belong to an apple in the image).
- __Non-real-time rendering__: Instead of having to move the Probe in real-time, perhaps for a longer recording, you can script a "timeline" and render it non-real-time. You can also reuse a script to render the same scan pattern on many images.
- __Headless mode__: While the `App` class is meant to help with interactive auditory-visual exploration, you can totally skip its entire graphical user interface, and control it using its properties. You should also use headless mode if you are outside of a Jupyter Notebook environment, and using Pixasonics in a script.
- __Remote control via OSC__: Since all of the `App` settings can be controlled via properties, while having the UI automatically update, it is possible to fully "remote-control" an `App` via the [Open Sound Control (OSC)](https://en.wikipedia.org/wiki/Open_Sound_Control) protocol.
- __Multichannel `Synth`s__: Providing a list instead of a number for any of a `Synth`s arguments will make it multichannel, which can be used to sonify `Feature`s that have more than one number. And don't worry if the number of features do not match the number of `Synth` channels: in this case `Mapper`s will dynamically resample the feature vector to fit the number of channels.
- __`Feature` base class and custom `Feature`s__: While there are lots of convenient abstractions for simple `Feature`s (e.g., `MeanChannelValue`, `MedianRowValue`, etc), these are all just configs for the `Feature` base class, and if you learn how it works, you can intuitively fit the `Feature` to whatever slice of the image you need to focus on, using any of the "built-in" (NumPy-based) reducing methods. But you can also create your completely custom `Feature` processors (let's say one that fits a K-means model on the image) by inheriting from the `Feature` base class and overriding two of its methods.
- __Custom `Synth`s from a SignalFlow `Patch`__: Since all audio processing in Pixasonics is based on the [SignalFlow](https://signalflow.dev/) library, it is also possible to create custom `Synth`s (that are fully compatible with the rest of Pixasonics) from a SignalFlow [`Patch`](https://signalflow.dev/patch/).
- __Multi-target mapping and custom `Mapper`s__: Sometimes you might want to link several `Synth` parameters (on the same or different `Synth`s) to the same `Feature`. In this case, you can create a single `Mapper` with a list of targets, while controlling the output ranges and exponents separately. Just like with `Feature`s and `Synth`s, it is also possible to create custom `Mapper`s.
- __Multiple `App`s in the same session__: You can also set up different `App`s with different pipelines (or images) and use them simultaneously in the same Notebook. For scientists, this can help testing the same sonifications on different images (or sequences), or different sonification setups on the same image data. For creatives, this will let you create different interactive instruments.

Before we jump into the advanced examples, let's import everything we'll need.

In [ ]:
import numpy as np
import os
from PIL import Image
from IPython.display import Audio, display
from pixasonics.core import App, Mapper, AudioIOSettings
from pixasonics.features import Feature, MeanChannelValue, MaxChannelValue
from pixasonics.synths import Theremin, Oscillator, FilteredNoise

## Loading HDR images and NumPy arrays

### HDR images from files

Pixasonics internally uses the [Pillow](https://pillow.readthedocs.io/en/stable/) library, which allows reading from a wide range of image formats and supports high bit rates! For an example, let's create an `App`:

In [ ]:
# Create app
app = App()

...then pass the path to a High Dynamic Range (HDR) image. Pixasonics happens to come with both the 8-bit and 16-bit versions of its example images (from the [CELLULAR open dataset](https://zenodo.org/records/8315423)), so you can start experimenting right away:

In [ ]:
# load HDR images
img_path = "images/cellular_dataset/single_channel_16bit/Timepoint_005_220518-ST_C03_s3_w1.TIF"
app.load_image_file(img_path)

Oh, something is wrong, the image is (almost completely) black! No, no, let me explain.

When you read a HDR image (in fact, any image), the `App` creates its own  image to display. This is so that the displayed image can be decoupled from the underlying data, and to make sure it is always 8-bit (which is a requirement for displaying it in a Jupyter Notebook — or a web browser). You can access this displayed image at any time from the `image_displayed` property. We can use the `display` function from IPython and the `Image` module from Pillow to create a static display of `app.image_displayed` in our Notebook:

In [ ]:
display(Image.fromarray(app.image_displayed))

Okay, but this is still quite dark. The reason for this is that in the input image the data did not span the full 16-bit range. This can often be the case with scientific imaging, where microscopes will record their raw values without stretching their ranges (i.e. normalizing them) to the available headroom. Now let's activate normalization in our `App` and have another snapshot of its `image_displayed`:

In [ ]:
app.normalize_display = True
display(Image.fromarray(app.image_displayed))

Now you can see that both inside our `App`'s UI and in the new snapshot the displayed image is normalized, so we can see those _Drosophila_ S2 cells much better. An important takeaway here is that nothing actually happened to the 16-bit image data that we read from the file (and which we can access using the `image` property, here "`app.image`"), the `App` merely re-created its displayed image. The previous image was so dark because the 8-bit displayed image was created only using the fact that the input image was 16-bit, so when creating its displayed image the `App` scaled the full 16-bit range (65,536 steps) into the 8-bit range (256 steps).

By the way, do you remember what the Low Percentile and High Percentile sliders do in the Display Settings? They can be especially handy with HDR images, such as this one. Try wiggling the sliders and see how the displayed image changes. Here are a few examples:

In [ ]:
# reveals darker details at the expense of saturating the brighter areas
app.display_normalization_low_percentile = 0
app.display_normalization_high_percentile = 80
display(Image.fromarray(app.image_displayed))

In [ ]:
# isolates very bright cells at the expense of losing darker details
app.display_normalization_low_percentile = 93
app.display_normalization_high_percentile = 94
display(Image.fromarray(app.image_displayed))

In [ ]:
# inverts the image
app.display_normalization_low_percentile = 100
app.display_normalization_high_percentile = 0
display(Image.fromarray(app.image_displayed))

Before we move on, let's set the low and high percentiles back to their defaults:

In [ ]:
app.display_normalization_low_percentile = 0
app.display_normalization_high_percentile = 100

### NumPy arrays

What is even better than reading HDR image files is the ability to load image data directly from a NumPy array! This opens up a lot of possibilities, such as:
- implementing any kind of preprocessing before loading the data into a Pixasonics `App`,
- procedurally creating image _volumes_, images with an arbitrary number of color channels or layers.

As briefly mentioned above, internally images have 4 dimensions in an `App`: __Height, Width, Channels, Layers__. Any of these dimensions can have any size your computer can handle. Let's see a few examples!

When you want to read from a NumPy array, you have to make sure that the array you pass to the `load_image_data` method has up-to 4 dimensions. An example of a 2D (only Height and Width) image:

In [ ]:
# load image as a numpy array
img_path = "images/cellular_dataset/merged_8bit/Timepoint_005_220518-ST_C03_s3.jpg"
img = Image.open(img_path)
img = np.array(img)
app.load_image_data(img) # load as numpy array

Next, let's read two images (that capture red and green fluorescent proteins) and stack them on the third (Channels) dimension, then load our (Height x Width x 2)-shaped image into the `App`. (Don't worry, it will automatically add an empty 3rd channel for its displayed image!)

In [ ]:
# combine two HDR images and load as numpy array
img_path = "images/cellular_dataset/single_channel_16bit/Timepoint_005_220518-ST_C03_s3_w2.TIF"
img_path2 = "images/cellular_dataset/single_channel_16bit/Timepoint_005_220518-ST_C03_s3_w1.TIF"
img = Image.open(img_path)
img2 = Image.open(img_path2)
img = np.array(img)
img2 = np.array(img2)
img = np.stack([img, img2], axis=-1)
print(img.shape)
app.load_image_data(img) # load as numpy array

Now the fun stuff: stacking image sequences in the Channel and Layer dimensions. This is the case where the Channel Offset and Layer Offset sliders (under Display Settings) become really useful. To showcase how you could "peek through" a hyperspectral image (with, for example, 128 color channels), let's read a time-lapse of 5 red channel images and stack them along the Channel dimension:

In [ ]:
# read all red-ch images into arrays and concatenate them in the channel dimension
img_folder = "images/cellular_dataset/single_channel_16bit/"
img_files = sorted(os.listdir(img_folder))
img_files = [f for f in img_files if f.endswith("w2.TIF")] # only red channel images
imgs = []
for img_file in img_files:
    img_path = os.path.join(img_folder, img_file)
    img = Image.open(img_path)
    img = np.array(img)
    imgs.append(img)
img = np.stack(imgs, axis=-1) # now the last dimension is the channel dimension
print(img.shape)
app.load_image_data(img) # load as numpy array

Now we can look through the image channels using a visualization of overlaying 3 consecutive channels and displaying them as RGB. Even when not working with hyperspectral images, this could be used to visualize camera drift (or cell migration), among other things. When `app.display_channel_offset` is 0, we look at an RGB visualization of channels [0, 1, 2], if the Channel Offset is 1, we look at channels [1, 2, 3] and so on.

If you want to flip through individual image channels (which here correspond to steps in time), you can set the maximum displayed channels to 1:

In [ ]:
app.maximum_displayed_channels = 1

For our current dataset (being a time lapse) it may make more sense to stack the images on the Layer dimension instead. We can do this just like before, the trick is only that we add an empty dimension to the 2D images, so their shape becomes (Height, Width, 1) before stacking:

In [ ]:
# read all red-ch images into arrays and concatenate them in the layer dimension
img_folder = "images/cellular_dataset/single_channel_16bit/"
img_files = sorted(os.listdir(img_folder))
img_files = [f for f in img_files if f.endswith("w2.TIF")] # only red channel images
imgs = []
for img_file in img_files:
    img_path = os.path.join(img_folder, img_file)
    img = Image.open(img_path)
    img = np.array(img)[..., None] # add a new dimension for channels
    imgs.append(img)
img = np.stack(imgs, axis=-1) # now the last dimension is the layer dimension
print(img.shape)
app.load_image_data(img) # load as numpy array

Now the Layer Offset slider got enabled so we can conveniently flip through the time lapse with it. Depending on your machine, and your `App`'s `image_size` (that is 500x500 by default) this could be a bit laggy, since every time the Layer (or Channel) offset is changed a new displayed image is rendered by the `App`. Notice also how the Channel Offset slider is disabled now, since our image volume has a single color channel.

Now let's stack the red and green component images on the Channel dimension while stacking the time steps on the Layer dimension, so our final shape is (Height, Width, 2, 5):

In [ ]:
# combine red and green channels and all layers
img_folder = "images/cellular_dataset/single_channel_16bit/"
img_files = sorted(os.listdir(img_folder))
imgs_red = [f for f in img_files if f.endswith("w2.TIF")] # only red channel images
imgs_green = [f for f in img_files if f.endswith("w1.TIF")] # only green channel images
imgs = []
for img_red, img_green in zip(imgs_red, imgs_green):
    img_path_red = os.path.join(img_folder, img_red)
    img_path_green = os.path.join(img_folder, img_green)
    img_red = Image.open(img_path_red)
    img_green = Image.open(img_path_green)
    img_red = np.array(img_red)
    img_green = np.array(img_green)
    img = np.stack([img_red, img_green], axis=-1) # now the last dimension is the channel dimension
    imgs.append(img)
img = np.stack(imgs, axis=-1) # now the last dimension is the layer dimension
print(img.shape)
app.load_image_data(img) # load as numpy array

(The snippet above is so handy that we'll re-use it in the following examples as well.)

Let's now forget to set our maximum displayed channels back to 3:

In [ ]:
app.maximum_displayed_channels = 3

Lastly, to use all images in the dataset, let's add "brightfield" image to the 3rd (blue) channel:

In [ ]:
# now use all images in the folder
img_folder = "images/cellular_dataset/single_channel_16bit/"
img_files = sorted(os.listdir(img_folder))
imgs_red = [f for f in img_files if f.endswith("w2.TIF")] # only red channel images
imgs_green = [f for f in img_files if f.endswith("w1.TIF")] # only green channel images
imgs_blue = [f for f in img_files if f.endswith("w3.TIF")] # only blue channel images
imgs = []
for img_red, img_green, img_blue in zip(imgs_red, imgs_green, imgs_blue):
    img_path_red = os.path.join(img_folder, img_red)
    img_path_green = os.path.join(img_folder, img_green)
    img_path_blue = os.path.join(img_folder, img_blue)
    img_red = Image.open(img_path_red)
    img_green = Image.open(img_path_green)
    img_blue = Image.open(img_path_blue)
    img_red = np.array(img_red)
    img_green = np.array(img_green)
    img_blue = np.array(img_blue)
    img = np.stack([img_red, img_green, img_blue], axis=-1) # now the last dimension is the channel dimension
    imgs.append(img)
img = np.stack(imgs, axis=-1) # now the last dimension is the layer dimension
print(img.shape)
app.load_image_data(img) # load as numpy array

This may have created a strong blue tint since the number range of the bright-field image is so different. We can undo that tint by enabling global normalization instead:

In [ ]:
app.normalize_display_global = True
display(Image.fromarray(app.image_displayed))

...or without global normalization, but via limiting the maximum displayed channels to 2:

In [ ]:
app.normalize_display_global = False
app.maximum_displayed_channels = 2
display(Image.fromarray(app.image_displayed))

## Probing with masks

Now let's look at how we can probe the image using instance masks. First, let's create a new `App`:

In [ ]:
app_2 = App()

Now let's load the full 16-bit image sequence with both red and green channels and all five time steps as before:

In [ ]:
# combine red and green channels and all layers
img_folder = "images/cellular_dataset/single_channel_16bit/"
img_files = sorted(os.listdir(img_folder))
imgs_red = [f for f in img_files if f.endswith("w2.TIF")] # only red channel images
imgs_green = [f for f in img_files if f.endswith("w1.TIF")] # only green channel images
imgs = []
for img_red, img_green in zip(imgs_red, imgs_green):
    img_path_red = os.path.join(img_folder, img_red)
    img_path_green = os.path.join(img_folder, img_green)
    img_red = Image.open(img_path_red)
    img_green = Image.open(img_path_green)
    img_red = np.array(img_red)
    img_green = np.array(img_green)
    img = np.stack([img_red, img_green], axis=-1) # now the last dimension is the channel dimension
    imgs.append(img)
img = np.stack(imgs, axis=-1) # now the last dimension is the layer dimension
print(img.shape)
app_2.load_image_data(img) # load as numpy array

Let's also turn on normalization and bring down the high percentile to a reasonable level to get a good view of most cells:

In [ ]:
app_2.display_normalization_high_percentile = 90
app_2.normalize_display = True

### Loading masks

Now let's load instance masks! Luckily, the [CELLULAR dataset](https://zenodo.org/records/8315423) also contains masks corresponding to individual *Drosophila* S2 cells in the images, some of which also bundled with Pixasonics. This is how you can load them:

In [ ]:
masks_folder = "images/cellular_dataset/masks/"
mask_files = sorted(os.listdir(masks_folder))
mask_files = [f for f in mask_files if f.endswith(".tif")] # only tif files
masks = []
for mask_file in mask_files:
    mask_path = os.path.join(masks_folder, mask_file)
    mask = Image.open(mask_path)
    mask = np.array(mask)
    masks.append(mask[..., None]) # add a new dimension for channels
mask = np.stack(masks, axis=-1) # now the last dimension is the layer dimension
print(mask.shape)
app_2.load_mask_data(mask) # load as numpy array

Let's also create a minimal sonification pipeline. This time we use the maximum red channel value as our sonified `Feature`. We map this to the center frequency of a bandpass filter applied to white noise. Notice how this time we specify a cubic curve from 100 Hz to 6000 Hz as we create the `Mapper`.

In [ ]:
max_red = MaxChannelValue(filter_channels=0, name="MaxRed")
app_2.attach(max_red)

filtered_noise = FilteredNoise(name="FilteredNoise")
filtered_noise.set_input("resonance", 0.9) # set filter resonance to 0.9
app_2.attach(filtered_noise)

red2noisefreq = Mapper(max_red, filtered_noise["cutoff"], out_low=100, out_high=6000, exponent=2, name="Red2NoiseFreq")
app_2.attach(red2noisefreq)

app_2.audio = True # activate audio output

Let's create a new copy of the `App`'s UI:

In [ ]:
app_2.ui

If you look into the Probe Settings, you should see that the "Probe with mask" check box is enabled (i.e., it is clickable)since we just loaded some mask data. Let's check the box:

In [ ]:
app_2.probe_with_mask = True

A handful of visual changes just happened. On the Probe Settings pane, two more check boxes (both checked by default) are enabled now, as well as a number box and a button. On the image canvas, depending on where your Probe was, you may see that it "snapped" onto a nearby object mask, drawing a bounding box around, and a semi-transparent highlight over the object (here, a cell).

If you don't see that, try to drag around the image (the `App` should currently be in Hold interaction mode, unless you changed that) and give it a listen. Notice that as you drag your mouse around, the Probe snaps to the object mask you hovered over. An important detail here is that the Probe will ignore hovering over the background (which conventionally has the 0 label) and only change selection when you hover over a non-zero mask ID. Check out also how the "Selected mask ID" number box updates on the Probe Settings pane as the selection changes. Try setting the number *through* the box: increment or decrement the value with the little arrows on the right, or type in another number and hit Enter. You can also set the value programmatically like this:

In [ ]:
app_2.selected_mask_ids = 42 # selects the object mask with the ID of 42

### Masks under the hood

How do masks communicate which ID belongs to which object and what the spatial dimensions of that object are? Masks are images where pixel values function as labels, or: identifiers (IDs) corresponding to individual objects in the image. Here is a little illustration:

In [ ]:
mask_id_to_highlight = 42 # try changing this number are re-evaluating this cell
layer = 0 # which layer of the mask we should look at (remember, there are 5 layers in this mask)
masked_img = app_2.mask[..., layer] == mask_id_to_highlight
masked_img = masked_img[..., 0] # remove the channel dimension (only leaving (Height, Width))
display(Image.fromarray(masked_img))

### Selecting objects via mousing

Let's discuss those check boxes. They change how mousing over the image (when "Probe with mask" is checked) is interpreted by the `App`. It comes down to the relationship of the numbers in your mask across different channels and layers and answering the question: *does the same ID mean the same object?* If so, clicking on the object in any view (at any display channel or layer offset) should select the object "everywhere": on all spatial or temporal slices. This way, the sonified sound you hear corresponds to the object across the whole image volume.

How is it in this mask dataset: does the mask ID of 42 (or any other ID) correspond to the same object across different layers? (By the way, since we loaded a mask volume with 1 channel, the `App` interprets this as the same mask ID being used for both the red and green color channels, which is what we want here.) To test this, select a mask ID, then open the Display Settings pane and use the Layer Offset slider to scroll through the time lapse.

As you probably noticed, the bounding box jumps around as you step through, which means the same mask ID *does not* correspond to the same object in this mask volume. This could be expected in an image dataset like CELLULAR, since having the IDs correspond to the same cells would require tracking a massive number of cells through time, which is not a trivial task.

So how can we select the same object across all layers in our image volume? This is where the "Same mask IDs..." check boxes come handy. When you uncheck the "Same mask IDs across layers" check box, the `App` will select *all* mask IDs across all layers under the mouse cursor. Imagine it as if, instead of selecting objects by their ID, you would use a thin (one-pixel) needle to pierce through the image volume. Whichever masks get hit will get selected.

Since, in this particular time lapse, cells do not migrate too much, clicking around the middle of a cell should select it (if it exists) on all time steps. Let's *not* use the same IDs across layers by unchecking the corresponding check box or evaluating:

In [ ]:
app_2.same_mask_ids_across_layers = False

Try to drag around your cursor within the same cell and watch what happens to the "Selected mask IDs" number box. Spoiler: it should turn into a textbox displaying an array of 5 numbers, corresponding the different mask IDs across the 5 layers of our image volume. Try to find a position where all 5 numbers are non-zero, then verify the selection by scrolling through the layers of the image (using the Layer Offset slider).

As you see, the "Same mask IDs..." options can be used as rudimentary "tracking", as long as the tracked object shares at least one pixel (at the same width and height coordinate) across channels or layers.

### Selecting multiple mask IDs

In [ ]:
app_2.ui # in case you want to save some scrolling...

What if you wanted to select several objects at once? It could be that you ran your awesome tracking algorithm and now have a list of IDs for each cell (corresponding to the same object at different times). The `selected_mask_ids` property can be set in a variety of ways. Here are a few examples:

In [ ]:
# select IDs 42 and 24 across all channels and layers
app_2.selected_mask_ids = [42, 24]

In [ ]:
# select 42 on layer 0 and 24 on layer 1, do not select anything on the other layers
app_2.selected_mask_ids = [[42, 24, 0, 0, 0]]

In [ ]:
# select 42 on channel 0 and 24 on channel 1, on all layers
app_2.selected_mask_ids = [[42, 42, 42, 42, 42], [24, 24, 24, 24, 24]]

In [ ]:
# on layer 0: select 2 on channel 0 and 4 on channel 1
# on layer 1: select 24 on channel 0 and 42 on channel 1
# on layer 4: select 42 on both channels
app_2.selected_mask_ids = [[2, 24, 0, 0, 42], [4, 42, 0, 0, 24]]

Here is an example of selecting the same cell tracked across all layers:

In [ ]:
app_2.selected_mask_ids = [[154, 168, 185, 140, 143]]

Making selections this way is not very intuitive. For making custom selections visually, let's use the "Select" interaction mode, which functions differently when probing with masks.

In [ ]:
app_2.interaction_mode = "select"

Now you can double click on objects to include or exclude them from the selection. You can also do this while listening to the output: for example, use Toggle interaction mode, and double click on an object to turn sound on, then switch to Select mode, and double click on other cells to include in or exclude them from the selection. Give it a try!

Note that, in this example sonification pipeline, we are using the *maximum* red value within the Probe as our image feature. This means, adding or removing a darker cell from the selection while a brighter cell is included will make no acoustic difference.

Finally, we can clear our selection by using the "Clear all selections" button, or evaluating:

In [ ]:
app_2.selected_mask_ids = None

## Non-real-time rendering

Previously we looked at how it is possible to make quick real-time recordings using the Record button (or the `recording` boolean property). It is also possible to render sonifications Non-Real-Time (NRT). We can do this by creating a "timeline", where we script the size and position of the `App`'s Probe with respect to time values. A timeline is expected to have the following structure:
- At the top-level the timeline a `List`. 
    - Each list element should be a `Tuple` of 
        - a `float`, which denotes the time in seconds,
        - and a `Dict`, that contains "settings" for that time point, where keys are the names of properties. Currently supported include: 
            - `"probe_width" (int)`, 
            - `"probe_height" (int)`, 
            - `"probe_x" (int)`, 
            - `"probe_y" (int)`,
            - `"filter_probe_by_channel_offset" (bool)`,
            - `"filter_probe_by_layer_offset" (bool)`,
            - `"display_channel_offset" (int)`,
            - `"display_layer_offset" (int)`,
            - `"probe_with_mask" (bool)`,
            - `"probe_with_mask" (List|np.array)`.

Let's see a toy example! First let's create another `App`:

In [ ]:
app_3 = App()

...then load all color channels and time steps of the image like before:

In [ ]:
# combine red and green channels and all layers
img_folder = "images/cellular_dataset/single_channel_16bit/"
img_files = sorted(os.listdir(img_folder))
imgs_red = [f for f in img_files if f.endswith("w2.TIF")] # only red channel images
imgs_green = [f for f in img_files if f.endswith("w1.TIF")] # only green channel images
imgs = []
for img_red, img_green in zip(imgs_red, imgs_green):
    img_path_red = os.path.join(img_folder, img_red)
    img_path_green = os.path.join(img_folder, img_green)
    img_red = Image.open(img_path_red)
    img_green = Image.open(img_path_green)
    img_red = np.array(img_red)
    img_green = np.array(img_green)
    img = np.stack([img_red, img_green], axis=-1) # now the last dimension is the channel dimension
    imgs.append(img)
img = np.stack(imgs, axis=-1) # now the last dimension is the layer dimension
print(img.shape)
app_3.load_image_data(img) # load as numpy array

(optionally enable normalization for a clear display:)

In [ ]:
app_3.normalize_display = True
app_3.display_normalization_low_percentile = 0
app_3.display_normalization_high_percentile = 90

...then create a minimal sonification setup (mapping the mean read pixel value under the Probe to the frequency of a sine wave):

In [ ]:
# example minimal setup

# create a feature that will report the mean value of the red channel
mean_red_3 = MeanChannelValue(filter_channels=0, name="MeanRed")
# attach the feature to the app
app_3.attach(mean_red_3)

# create a Theremin, a simple sine wave synth that we will use to sonify the mean pixel value
theremin_3 = Theremin()
# attach the Theremin to the app
app_3.attach(theremin_3)

# create a Mapper that will map the mean red pixel value (within the Probe) to the frequency of the Theremin
red2freq_3 = Mapper(
    mean_red_3, 
    theremin_3["frequency"], 
    in_low=200,
    in_high=1000,
    exponent=2, # cubic mapping curve for a more "linear" feel of frequency changes
    name="Red2Freq")
# attach the Mapper to the app
app_3.attach(red2freq_3)

Rendering a timeline is essentially scripting the available settings (which are `App` properties) through time. In the example below we specify only two timepoints: the start (at 0 s) and the end (specified in the `duration` variable). This example shows how to perform a "horizontal scan", where the Probe is a vertical line (its width being 1 pixel and height being the image size) and placed on the left edge of the image (since the Probe height matches the image height, the value of `probe_y` does not matter). In the second timepoint, we specify that the Probe should be at the right edge of the image.

Property values that are not named will repeat their last specified values. Numerical values between adjacent timepoints linearly interpolate while boolean values (as well as `"selected_mask_ids"`) will function as triggers: they modify the state at the prescribed time without any interpolation. This example will render a *horizontal_scan.wav* file with a duration of 5 seconds. We can use `display` and `Audio` from IPython to embed the result in the Notebook.

In [ ]:
# example: horizontal scan
duration = 5
my_timeline = [
    (0, {
        "probe_width": 1, # horizontal scanning (a vertical scan line)
        "probe_height": 500,
        "probe_x": 0, # start at the left edge of the image
        "probe_y": 0
    }),
    (duration, {
        "probe_x": 499 # move the probe to the right edge of the image
    })
]

target_filename = "horizontal_scan.wav"

app_3.render_timeline_to_file(my_timeline, target_filename)

display(Audio(target_filename)) # display the resulting audio file

(Ignore the "*Warning: buffer overrun?*" message from SignalFlow, this is spurious, and will get changed in future versions.)

Sounds squeaky! It did not take long for this short file, but you may want to render files much longer. In that case, it is useful to know about the `fps` property of the `App`, which specifies how many times a second the whole sonification pipeline is evaluated. Its default value is `60`, which means all `Feature`s and `Mapper`s are evaluated 60 times a second. When rendering long sonificaitons, you may get away with a perceptually identical result with much less rendering time if you temporarily reduce the `fps`, like in the following example:

In [ ]:
# example: long horizontal scan, so we reduce app fps to optimize rendering
app_2.fps = 10 # reduce fps to 10 for longer duration rendering
duration = 60 # 60 seconds of audio
my_timeline = [
    (0, {
        "probe_width": 1, # horizontal scanning (a vertical scan line)
        "probe_height": 500,
        "probe_x": 0,
        "probe_y": 0
    }),
    (duration, {
        "probe_x": 499
    })
]

target_filename = "long_horizontal_scan.wav"

app_3.render_timeline_to_file(my_timeline, target_filename)

display(Audio(target_filename))
app_3.fps = 60 # restore fps to 60 for interactive use

Can you guess how a vertical scan (with a horizontal scan line) setup would look like? Take a look below: it is almost the same as our first example, except that we set the *height* of the Probe to 1 pixel.

(By the way, nothing stops you from changing the size of the Probe between timepoints!)

In [ ]:
# example: vertical scan
duration = 5
my_timeline = [
    (0, {
        "probe_width": 500,
        "probe_height": 1, # vertical scanning (a horizontal scan line)
        "probe_x": 0,
        "probe_y": 0
    }),
    (duration, {
        "probe_y": 499
    })
]

target_filename = "vertical_scan.wav"

app_3.render_timeline_to_file(my_timeline, target_filename)

display(Audio(target_filename))

For further examples, let's also load the corresponding instance masks, like before:

In [ ]:
masks_folder = "images/cellular_dataset/masks/"
mask_files = sorted(os.listdir(masks_folder))
mask_files = [f for f in mask_files if f.endswith(".tif")] # only tif files
masks = []
for mask_file in mask_files:
    mask_path = os.path.join(masks_folder, mask_file)
    mask = Image.open(mask_path)
    mask = np.array(mask)
    masks.append(mask[..., None]) # add a new dimension for channels
mask = np.stack(masks, axis=-1) # now the last dimension is the layer dimension
print(mask.shape)
app_3.load_mask_data(mask) # load as numpy array

In the following example, we showcase how to use masks and `"filter_probe_by_layer_offset"` in combination with `"display_layer_offset"`. For masks, we load the IDs corresponding to the same cell across all layers (here: time steps). In a real-time scenario, this would let you hear how the currently used `Feature` (mean red pixel value within the Probe) changes as you drag the Layer Offset slider.

In [ ]:
# example: timelapse of a single cell across all layers (time points)
duration = 1
num_layers = app_3.image.shape[-1]
my_timeline = [
    (0, {
        "probe_with_mask": True,
        # select the IDs corresponding to the same cell across all layers (here: time steps)
        "selected_mask_ids": [[73, 67, 122, 83, 16]],
        "filter_probe_by_layer_offset": True,
        "display_layer_offset": 0,
    }),
    (duration * 0.8, {
        "display_layer_offset": num_layers - 1
    }),
    (duration, {
        "display_layer_offset": num_layers - 1
    })
]

target_filename = "cell_redness_timelapse.wav"

app_3.render_timeline_to_file(my_timeline, target_filename)

display(Audio(target_filename))

The following example demonstrates how you can change selected mask IDs or even the Probing method within the same timeline. Notice how we also define time values as fractions of the `duration`, so we can easily stretch or shrink the whole timeline based on the total duration of the result.

We start out using masks, switch between different selections, and just before the recording terminates, we switch back to the "normal" rectangular Probe and perform a diagonal scan.

In [ ]:
# example: mix between Probing with and without masks
duration = 10 # try changing this
my_favorite_ids = [42, 24, 2, 4]
my_timeline = [
    (0, {
        "probe_with_mask": True,
        "selected_mask_ids": 1,
    }),
    (duration * 0.1, {
        "selected_mask_ids": 20
    }),
    (duration * 0.2, {
        "selected_mask_ids": [2, 3]
    }),
    (duration * 0.4, {
        "selected_mask_ids": my_favorite_ids
    }),
    (duration * 0.6, {
        "selected_mask_ids": 1
    }),
    (duration * 0.8, {
        "probe_with_mask": False,
        "probe_x": 0,
        "probe_y": 0,
    }),
    (duration, {
        "probe_x": 500,
        "probe_y": 500,
    })
]

target_filename = "mix_and_match.wav"

app_3.render_timeline_to_file(my_timeline, target_filename)

display(Audio(target_filename))

## Headless mode

What if you want the real-time sonification pipeline, but don't need the graphical interface? Maybe you want to build your own UI, or just live code? Then headless mode is for you!

Since it is possible to control `App`s via properties, you can initialize them with `headless=True` to skip rendering the built-in UI entirely. Let's try it:

In [ ]:
app_4 = App(headless=True) # create a new app in headless mode (no UI)

That's right, there is no UI popping up. But our `App` is still ready. Let's load an image into it:

In [ ]:
app_4.load_image_file("images/test.jpg")

...and then add a minimal sonification pipeline:

In [ ]:
mean_red_4 = MeanChannelValue(filter_channels=0, name="MeanRed")
app_4.attach(mean_red_4)
theremin_4 = Theremin(name="MySine")
app_4.attach(theremin_4)
red2freq_4 = Mapper(mean_red_4, theremin_4["frequency"], exponent=2, name="Red2Freq")
app_4.attach(red2freq_4)
app_4.features, app_4.synths, app_4.mappers # check that the app has the expected features, synths and mappers

Upon checking the `features`, `synths`, and `mappers` properties, we see that the pipeline is there. Similarly, we can check the current size and position of the Probe:

In [ ]:
app_4.probe_width, app_4.probe_height, app_4.probe_x, app_4.probe_y

Let's move it away from the top left corner:

(feel free to evaluate the cell above after you evaluated the cell below to check)

In [ ]:
app_4.probe_x, app_4.probe_y = 200, 200

Now let's enable audio in the `App`:

In [ ]:
app_4.audio = True

We can always access the global audio graph from any `App`'s `graph` property. This will return the `AudioGraph` singleton object from SignalFlow. Let's check the `status` of the graph:

In [ ]:
app_4.graph.status

Note that the output is silent. Now let's unmute the audio (equivalent to activating the Probe on the visual display) and then check the graph `status` again. (You should also hear a sine tone.)

In [ ]:
app_4.unmuted = True

Let's move the Probe again, and listen if the sound changes. (Be careful with your speaker levels!)

In [ ]:
app_4.probe_x, app_4.probe_y = 50, 400

(Auch!) It changed indeed, now let's mute the `App` (equivalent to deactivating the Probe) again:

In [ ]:
app_4.unmuted = False

Here is a silly little snippet to move the Probe real-time while it is unmuted. Since we loaded *test.jpg* up top, which is a simple gradient, you should hear an upwards glide.

In [ ]:
# a little loop to unmute the probe, then move it around then turn it off
import time
app_4.unmuted = True
app_4.probe_x, app_4.probe_y = 250, 0
while app_4.probe_y < 350:
    app_4.probe_y += 1
    time.sleep(0.01)
app_4.unmuted = False

Now let's swap the image and then try evaluating the snippet above again!

In [ ]:
app_4.load_image_file("images/cellular_dataset/merged_8bit/Timepoint_001_220518-ST_C03_s3.jpg")

Hopefully that gives you an idea about headless mode. While it is optional when you are working in a Notebook, it is crucial when you are working in an offline script. Here is an example script that gets all image files in a folder and then renders the same horizontal scanning sonification path on all images in a loop:

In [ ]:
# a for loop where for each image in the folder we load a headless app and render a timeline in nrt mode
img_folder = "images/cellular_dataset/merged_8bit/"
img_files = sorted(os.listdir(img_folder))
img_files = [f for f in img_files if f.endswith(".jpg")] # only jpg files

# example: horizontal scan
duration = 5
my_timeline = [
    (0, {
        "probe_width": 1,
        "probe_height": 500,
        "probe_x": 0,
        "probe_y": 0
    }),
    (duration, {
        "probe_x": 499
    })
]
_app = App(headless=True, nrt=True) # create an App
# only need to create processor objects once and attach them to the App
_mean_red = MeanChannelValue(filter_channels=0, name="MeanRed") # we will attach it later, when there is an image loaded
_theremin = Theremin(name="MySine")
_red2freq = Mapper(_mean_red, _theremin["frequency"], exponent=2, name="Red2Freq")
# these we can already attach
_app.attach(_theremin)
_app.attach(_red2freq)
# loop over all images in the folder, load the image, and render the timeline
for img_file in img_files:
    print(f"Processing {img_file}")
    img_path = os.path.join(img_folder, img_file)
    _app.load_image_file(img_path)
    _app.attach(_mean_red) # attach the feature to the app when there is some image loaded. it will only happen the first time, and will be ignored after
    target_filename = img_file.replace(".jpg", ".wav")
    _app.render_timeline_to_file(my_timeline, target_filename)
    print(f"Saved {target_filename}")
    display(Audio(target_filename))

The above process rendered wave files via the `render_timeline_to_file` method. You might render sonifications as part of a larger workflow (maybe training a neural net to couple sound with images?), and in that case you may want to render sonifications to NumPy arrays instead via the `render_timeline_to_array` method. Here we still go for displaying these arrays as sound buffers:

In [ ]:
# same thing but render np.arrays instead
img_folder = "images/cellular_dataset/merged_8bit/"
img_files = sorted(os.listdir(img_folder))
img_files = [f for f in img_files if f.endswith(".jpg")] # only jpg files

# example: horizontal scan
duration = 5
my_timeline = [
    (0, {
        "probe_width": 1,
        "probe_height": 500,
        "probe_x": 0,
        "probe_y": 0
    }),
    (duration, {
        "probe_x": 499
    })
]
_app = App(headless=True, nrt=True) # create an App
# only need to create processor objects once and attach them to the App
_mean_red = MeanChannelValue(filter_channels=0, name="MeanRed") # we will attach it later, when there is an image loaded
_theremin = Theremin(name="MySine")
_red2freq = Mapper(_mean_red, _theremin["frequency"], exponent=2, name="Red2Freq")
# these we can already attach
_app.attach(_theremin)
_app.attach(_red2freq)
# loop over all images in the folder, load the image, and render the timeline
for img_file in img_files:
    print(f"Processing {img_file}")
    img_path = os.path.join(img_folder, img_file)
    _app.load_image_file(img_path)
    _app.attach(_mean_red) # attach the feature to the app when there is some image loaded. it will only happen the first time, and will be ignored after
    buf = _app.render_timeline_to_array(my_timeline)
    display(Audio(buf, rate=_app.sample_rate, normalize=False))

## Remote control via OSC

Since `App`s can be controlled via their properties, you could also include your sonification pipeline(s) in a broader setup via Open Sound Control (OSC). The following is a minimal example to help you get started.

First, you need to install the `python-osc` package (uncomment and evaluate the line below).

In [ ]:
# !pip install python-osc

Let's import the components we need from `python-osc`:

In [ ]:
from pythonosc import dispatcher, osc_server

Here is yet another instance of our toy sonification setup:

In [ ]:
# toy setup...
app_5 = App()
app_5.load_image_file("images/test.jpg")
mean_red_5 = MeanChannelValue(filter_channels=0, name="MeanRed")
app_5.attach(mean_red_5)
theremin_5 = Theremin(name="MySine")
app_5.attach(theremin_5)
red2freq_5 = Mapper(mean_red_5, theremin_5["frequency"], exponent=2, name="Red2Freq")
app_5.attach(red2freq_5)
app_5.audio = True # already start the audio graph

Now, we have to define all the helper functions ("the API") through which our OSC messages will be routed (by a `Dispatcher`) to `App` properties:

In [ ]:
# create helper functions
def handle_unmuted(address, value):
    print(f"Unmuted: {value}")
    app_5.unmuted = True if value > 0 else False
def handle_probe_x(address, value):
    print(f"Probe X: {value}")
    app_5.probe_x = value
def handle_probe_y(address, value):
    print(f"Probe Y: {value}")
    app_5.probe_y = value
def handle_probe_xy(address, x, y):
    print(f"Probe XY: {x}, {y}")
    app_5.probe_x = x
    app_5.probe_y = y
def handle_probe_width(address, value):
    print(f"Probe Width: {value}")
    app_5.probe_width = value
def handle_probe_height(address, value):
    print(f"Probe Height: {value}")
    app_5.probe_height = value

Then we create the `Dispatcher` and set up the map between OSC addresses and these helper functions:

In [ ]:
# create a Dispatcher and map addresses to functions
my_dispatcher = dispatcher.Dispatcher()
my_dispatcher.map("/unmuted", handle_unmuted)
my_dispatcher.map("/probe/x", handle_probe_x)
my_dispatcher.map("/probe/y", handle_probe_y)
my_dispatcher.map("/probe/xy", handle_probe_xy)
my_dispatcher.map("/probe/w", handle_probe_width)
my_dispatcher.map("/probe/h", handle_probe_height)

Finally, let's launch the OSC server. We will launch the server in another thread to avoid blocking the notebook here.

In [ ]:
# create a server (using threading for non-blocking operation)
ip = "127.0.0.1"
port = 12345
_osc_server = None # keep a handle to the server
_server_thread = None # keep a handle to the server thread

def start_osc_server(ip, port, dispatcher):
    """Starts the OSC server in a separate thread."""
    global _osc_server, _server_thread
    if _osc_server:
        print("OSC server is already running.")
        return
    
    _osc_server = osc_server.ThreadingOSCUDPServer((ip, port), dispatcher)
    print(f"Serving on {_osc_server.server_address}")
    # Start a thread with the server -- that thread will then start one
    # more thread for each request
    import threading
    _server_thread = threading.Thread(target=_osc_server.serve_forever)
    _server_thread.daemon = True
    _server_thread.start()

def stop_osc_server():
    """Stops the running OSC server."""
    global _osc_server, _server_thread
    if _osc_server:
        print("Stopping OSC server...")
        _osc_server.shutdown()
        _server_thread.join()
        _osc_server = None
        _server_thread = None
        print("OSC server stopped.")
    else:
        print("OSC server is not running.")

# Start the server
start_osc_server(ip, port, my_dispatcher)

On the other side, you could set up another app to *send* OSC messages to control our sonification `App`. Here is a simple exampe in Max:

<img src="figures/osc_example_max_patch.png" alt="drawing" width="800"/>

Here is the compressed patcher you see above, in case you want to give it a try. You need to copy the cell below, then, in Max go to *File* > *New From Clipboard*.

```
<pre><code>
----------begin_max5_patcher----------
973.3oc0X0zbaBCD8r8uBUtzC0IAILXSu0+AY50NYxHiUsUFPhAIbbZl7eu5
CHAmXCJFhiyEX75EV81c0aehGGOxaAeKQ3A9I3OfQidb7nQFSZCip98HuL71
jTrv3lWBOKivjdSr+mjrUZrKHRv0E7EDvZBc05mcHGKSVSYqtsfjHsAJZV3k
9S.APys34leDeoO3lpmgtz7J4Kt6Bz75WDqLixRIRyx.VYbQ4hEojlVTtwKk
094uieB5RiuHswmFOVeYhi3Vjpd3h1PUjuAH1av.C3P9Mfkf9OS3C80tT+hJ
vYDIo3VBCWAE+8jFhaMMrKnqsZMIeHmXWgddfaNBjyH2qVBuof+C.zgrQr4F
J9xP0Uz9qwA96Ebn2G3nplxiCekYKboxNEYtEZQTSr3XQL.9NJhnCVDm.7Vf
YqFzhYdAImvVBtJWuE9p0NjMr4An+aSGMgL5yqusalp6oKksA0PneShpfVIp
l8UgnZZb3GHQU3WLhp5rgKDUnfyahpJrzahJj+WEhp6cHaTSTgZinBN+7jnJ
mlHs6nAEpbIQ.jbfbMA7q77uK.zL7Jxsl8qUOcJkQR3kLYyZydxNASMYmJR.
aNJX1ARN8jaa.yIY7MDK6cKUdq9RDpAsMZ5AfV3G.scv.BXIe0pTBnjkUJIK
aAzHjsD5vrJ3zOhYU6XMmSssfpFqga2e4xbgd2eJOAmtlKj.HR0H2RVwN5tZ
PNbZ6b.uGwJGCpr0xNWsU725V1W035H+MDNDjYG+bpNnt6tWFtKycTaUsyuS
F8pYTaenMjNeW0zgsA04mcP8N01bGPGxAwUyFBsUCJ391EJMDwsovvG1fyUy
1dX3E8oJc7EYDcWsf0zOubyIBHIOeCNszXSk21ygAaMETn+pQ64E3nZScpYR
OxPGZ96uIYbIAjvYxBdJ.CtltEK3LZhPqAqSRL60fJ0mG5vistw9upXWejLz
KmHq+yivRYQIs6Nh3lhDOlCTztdC8xv3ZiVfSw1dWgeTegenav+4MP8A7l0l
4T.u5y5Zvm19tYDAurHo9UUMOE7BBWRDRJCKobVCezJkzNs2jtqAZ9oJPybH
PyGf3D4PbzqEXOiSnCwIX.vyzSTbBPmpNA8WfsaHMHQx2kHAGhH4XW2PDGXG
wIZ.hCxEBn5za+hjKajPCQGNxkhDZPvjK8cvgfsCdx3ugtrqM9MAxNIDmmug
THpb1DCkJf63lQtFMldpYx1eZFw5UP1Pq82LL2CWnTFHUxBJKLqKusQVEMdY
bkjZlRDg9oGqQmJjFEFLkv.QN1BDiPjwOM9+MkkxtC
-----------end_max5_patcher-----------
</code></pre>
```

When done, shut down the OSC server like so:

In [ ]:
stop_osc_server() # call this to stop the server

## Multichannel Synths

One of the most exciting features of Pixasonics is the ability to work with multichannel `Synth`s. Every parameter of a `Synth` can be defined as a list of values, instead of a single number. In this case the `Synth` will grow additional channels to synthesize all parameters, and mix its output to stereo (2 channels).

Multichannel `Synth`s can be useful to sonify a list of `Feature`s as a distribution of a parameter. For example pixel intensities could be mapped to individual frequencies in an oscillator bank.

Note that multichannel expansion in Pixasonics does not behave identically to how it does in SignalFlow. While, in the latter, having a 2-channel parameter and a 11-channel parameter would mean repeating the first, 2-channel parameter to meet the channel count of the other one ("tiling": repeating the signal [1, 2] to become [1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1]), while in Pixasonics `Synth`s, the parameter with less channels will *interpolate* its channels (the signal [1, 2] becomes [1, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2]).

Additionally, `Mapper`s can dynamically project between a `Feature` vector and the number of channels in a `Synth`, so even if the length of the `Feature` vector changes over time (which we will demonstrate in a bit) there is no need to redo the mapping.

Let's see how all this works in action. First we create an `App`:

In [ ]:
app_6 = App()

Then load our usual image sequence:

In [ ]:
# combine red and green channels and all layers
img_folder = "images/cellular_dataset/single_channel_16bit/"
img_files = sorted(os.listdir(img_folder))
imgs_red = [f for f in img_files if f.endswith("w2.TIF")] # only red channel images
imgs_green = [f for f in img_files if f.endswith("w1.TIF")] # only green channel images
imgs = []
for img_red, img_green in zip(imgs_red, imgs_green):
    img_path_red = os.path.join(img_folder, img_red)
    img_path_green = os.path.join(img_folder, img_green)
    img_red = Image.open(img_path_red)
    img_green = Image.open(img_path_green)
    img_red = np.array(img_red)
    img_green = np.array(img_green)
    img = np.stack([img_red, img_green], axis=-1) # now the last dimension is the channel dimension
    imgs.append(img)
img = np.stack(imgs, axis=-1) # now the last dimension is the layer dimension
print(img.shape)
app_6.load_image_data(img) # load as numpy array
app_6.normalize_display = True # normalize the displayed image

Now let's create a pair of `Feature`s. The first one will reduce the Probe rectangle into a *column* (vertical line) by taking the average value of every row:

In [ ]:
# Mean row value (will change the number of features depending of the Probe height)
mean_row_6 = Feature(
    name="Mean Row",
    target_dim=0, # mean value of each row (for horizontal scanning)
)
app_6.attach(mean_row_6)

The second `Feature` will reduce the rectangle into a single *row* (horizontal line) by averaging columns:

In [ ]:
# Mean col value (will change the number of features depending of the Probe width)
mean_col_6 = Feature(
    name="Mean Col",
    target_dim=1, # mean value of each col (for horizontal scanning)
)
app_6.attach(mean_col_6)

Take a moment to try these out. Check their indicated number of features. Now go to the Probe Settings pane and change the Probe dimensions. Then check the `Feature`s again.

You probably noticed that the number of reported features changes according to the Probe width of height. Now let's create those oscillator banks we will use to listen to these changing feature vectors. We will use the `Oscillator` built-in `Synth`, and create two 16-channel instances, where the frequencies will be initialized with lists (of 16 elements), instead of single values.

Here is one:

In [ ]:
# create a 16-voice oscillator bank of sawtooth waves, spread across the stereo field
num_voices = 16
freqs = [440 for i in range(num_voices)]
osc_bank_6 = Oscillator(frequency=freqs, waveform="saw", name="OSCBank") # test multichannel
app_6.attach(osc_bank_6)

And the other one:

In [ ]:
# create a 16-voice oscillator bank of sawtooth waves, spread across the stereo field
num_voices = 16
freqs = [440 for i in range(num_voices)]
osc_bank_6_2 = Oscillator(frequency=freqs, waveform="saw", name="OSCBank2") # test multichannel
app_6.attach(osc_bank_6_2)

Take a moment to look at the UI cards of these `Synth`s. Instead of sliders, they now have text fields, indicating the current parameter values. Note that the *panning* parameter automatically created a spread (between -1 and 1, corresponding to hard-left and hard-right) populating the `Synth` channels in the stereo field from left to right.

Now the fun part: the mapping. Let's map both the Mean Row and Mean Col features to corresponding oscillator banks, so that the distribution of frequencies reflect the distribution of mean pixel values. Actually, we don't have to do anything tricky, we can use the `Mapper`s just like we did before, and they will automatically project between the number of features and the number of `Synth` channels.

Here is a `Mapper` mapping the Mean Row `Feature` vector to the frequencies of `osc_bank_6`:

In [ ]:
# Create a mapping where each row value will be mapped to a different channel of the oscillator bank
row2freq_6 = Mapper(mean_row_6, osc_bank_6["frequency"], exponent=2, out_high=1000, name="Row2Freq")
app_6.attach(row2freq_6)

...and another `Mapper` bridging the Mean Col `Feature` and `osc_bank_6_2`'s frequencies:

In [ ]:
# Create a mapping where each col value will be mapped to a different channel of the oscillator bank
col2freq_6 = Mapper(mean_col_6, osc_bank_6_2["frequency"], exponent=2, out_high=1000, name="Col2Freq")
app_6.attach(col2freq_6)

Don't forget to enable audio processing for the `App`:

In [ ]:
app_6.audio = True # enable audio output

Try it out! Sounds wild, doesn't it? You can use this setup to "listen through" all time steps (layers) in the image, to, for example, find cells that are hidden on the first time step. Can you find one like that? Try smaller Probe sizes so you can better focus on an area.

## The Feature base class and custom Features

### Feature base class

In [ ]:
app_7 = App()

In [ ]:
# combine red and green channels and all layers
img_folder = "images/cellular_dataset/single_channel_16bit/"
img_files = sorted(os.listdir(img_folder))
imgs_red = [f for f in img_files if f.endswith("w2.TIF")] # only red channel images
imgs_green = [f for f in img_files if f.endswith("w1.TIF")] # only green channel images
imgs = []
for img_red, img_green in zip(imgs_red, imgs_green):
    img_path_red = os.path.join(img_folder, img_red)
    img_path_green = os.path.join(img_folder, img_green)
    img_red = Image.open(img_path_red)
    img_green = Image.open(img_path_green)
    img_red = np.array(img_red)
    img_green = np.array(img_green)
    img = np.stack([img_red, img_green], axis=-1) # now the last dimension is the channel dimension
    imgs.append(img)
img = np.stack(imgs, axis=-1) # now the last dimension is the layer dimension
print(img.shape)
app_7.load_image_data(img) # load as numpy array

In [ ]:
# defaults
base_feature_7 = Feature(
    filter_rows=None,
    filter_columns=None,
    filter_channels=None,
    filter_layers=None,
    target_dim=2, # channels dimension
    reduce_method="mean",
    name="Feature"
)
app_7.attach(base_feature_7)

In [ ]:
# Mean pixel value
mean_pix_7 = Feature(name="Mean") # defaults to mean pixel value, channel dimension is kept
app_7.attach(mean_pix_7)

In [ ]:
# Median pixel value
median_pix_7 = Feature(name="Median", reduce_method="median")
app_7.attach(median_pix_7)

In [ ]:
# Mean row value (will change the number of features depending of the Probe height)
mean_row_7 = Feature(
    name="Mean Row",
    target_dim=0, # mean value of each row (for horizontal scanning)
)
app_7.attach(mean_row_7)

In [ ]:
# Mean col value (will change the number of features depending of the Probe width)
mean_col_7 = Feature(
    name="Mean Col",
    target_dim=1, # mean value of each col (for horizontal scanning)
)
app_7.attach(mean_col_7)

In [ ]:
# Median, filter for the red (first) channel only
median_red_7 = Feature(
    name="Median Red",
    target_dim=2, # channel dim
    filter_channels=0, # only use the first channel (red)
    reduce_method="median"
)
app_7.attach(median_red_7)

### Custom Features

In [ ]:
app_8 = App()

In [ ]:
# combine red and green channels and all layers
img_folder = "images/cellular_dataset/single_channel_16bit/"
img_files = sorted(os.listdir(img_folder))
imgs_red = [f for f in img_files if f.endswith("w2.TIF")] # only red channel images
imgs_green = [f for f in img_files if f.endswith("w1.TIF")] # only green channel images
imgs = []
for img_red, img_green in zip(imgs_red, imgs_green):
    img_path_red = os.path.join(img_folder, img_red)
    img_path_green = os.path.join(img_folder, img_green)
    img_red = Image.open(img_path_red)
    img_green = Image.open(img_path_green)
    img_red = np.array(img_red)
    img_green = np.array(img_green)
    img = np.stack([img_red, img_green], axis=-1) # now the last dimension is the channel dimension
    imgs.append(img)
img = np.stack(imgs, axis=-1) # now the last dimension is the layer dimension
print(img.shape)
app_8.load_image_data(img) # load as numpy array

In [ ]:
# create a custom feature that will return random values
class MyRandomFeature(Feature):
    def __init__(self, name="MyRandomFeature"):
        super().__init__(name=name)

    def process_image(self, mat):
        return np.random.rand(*mat.shape)
    
    def compute(self, mat):
        num_features = mat.shape[self.target_dim]
        return np.random.rand(num_features)
    
my_random_feature = MyRandomFeature()
app_8.attach(my_random_feature)

In [ ]:
# change target dim to height (0)
my_random_feature.target_dim = 0

#### K-means example

In [ ]:
# !pip install scikit-learn

In [ ]:
# create a custom feature that will fit a KMeans model to the image and return the histogram of cluster assignments
from sklearn.cluster import KMeans # pip install scikit-learn if you don't have it

class KMeansFeature(Feature):
    def __init__(self, n_clusters=3, name="KMeansFeature"):
        super().__init__(name=name)
        self.n_clusters = n_clusters
        self.kmeans = None
        self._original_shape = None

    def _reshape_for_kmeans(self, mat):
        """Helper to reshape 4D matrix to 2D for KMeans"""
        mat_reshaped = np.moveaxis(mat, self.target_dim, 0)
        return mat_reshaped.reshape(mat_reshaped.shape[0], -1)

    def process_image(self, mat):
        self._original_shape = mat.shape
        features = self._reshape_for_kmeans(mat)
        self.kmeans = KMeans(n_clusters=self.n_clusters).fit(features.T)

        # Get cluster assignments and reshape back to original dimensions
        labels = self.kmeans.predict(features.T)
        other_dims = [s for i, s in enumerate(mat.shape) if i != self.target_dim]
        self.transformed_image = np.expand_dims(
            labels.reshape(*other_dims), 
            axis=self.target_dim
        )
        return self.transformed_image
    
    def compute(self, mat):
        if self.kmeans is None:
            raise ValueError("KMeans model has not been fitted. Call process_image first.")
        features = self._reshape_for_kmeans(mat)
        labels = self.kmeans.predict(features.T)
        # Compute histogram of cluster assignments
        hist, _ = np.histogram(labels, bins=range(self.n_clusters + 1))
        return hist.astype(float) / hist.sum() # normalize to sum to 1

kmeans_feature = KMeansFeature(n_clusters=10)
app_8.attach(kmeans_feature)

In [ ]:
# create a multichannel Theremin that has its frequencies in a harmonic series
fundamental_freq = 110
num_harmonics = kmeans_feature.n_clusters
freqs = fundamental_freq * np.arange(1, num_harmonics + 1)
print("Frequencies:",freqs)
osc_8 = Theremin(frequency=freqs, name="KMeansOsc")
app_8.attach(osc_8)

# create a Mapper that will map the KMeans cluster histogram to the amplitude of the Theremin 
k2amp = Mapper(kmeans_feature, osc_8["amplitude"], exponent=1, name="K2Amp")
app_8.attach(k2amp)

In [ ]:
# !pip install matplotlib seaborn

In [ ]:
# visualize the KMeans clusters
import matplotlib.pyplot as plt # pip install matplotlib seaborn if you don't have them
import seaborn as sns

# Create a colormap with distinct colors for each cluster
n_clusters = kmeans_feature.n_clusters
colors = sns.color_palette("husl", n_colors=n_clusters)
colormap = {i: colors[i] for i in range(n_clusters)}

# Get the cluster assignments from transformed_image
cluster_image = kmeans_feature.transformed_image[:, :, 0, 0]
print(cluster_image.shape)

# Create RGB image where each cluster gets a unique color
rgb_image = np.zeros((*cluster_image.shape, 3))
for cluster_id, color in colormap.items():
    mask = cluster_image == cluster_id
    rgb_image[mask] = color

# Plot the results
plt.figure(figsize=(10, 5))

plt.subplot(121)
plt.title('Original Image')
plt.imshow(app_8.image_displayed)  # Show the currently displayed image
plt.axis('off')

plt.subplot(122)
plt.title('K-means Clusters')
plt.imshow(rgb_image)
plt.axis('off')

plt.tight_layout()
plt.show()

## Custom synths using the Synth base class

In [ ]:
import signalflow as sf
from pixasonics.synths import Synth

In [ ]:
app_9 = App()
app_9.audio = True
app_9.interaction_mode = "toggle"

In [ ]:
# define the SignalFlow patch as a subclass of sf.Patch (usual procedure)
class SimpleAMPatch(sf.Patch):
    def __init__(self, carr_freq=440, mod_freq=1, mod_depth=0.25):
        super().__init__()
        # define params
        carr_freq = self.add_input("carrier_freq", carr_freq)
        mod_freq = self.add_input("mod_freq", mod_freq)
        mod_depth = self.add_input("mod_depth", mod_depth)
        # build the patch for Amplitude Modulation
        modulator = (sf.SineOscillator(mod_freq) * mod_depth) + (1 - mod_depth)
        carrier = sf.SineOscillator(carr_freq)
        out = carrier * modulator
        self.set_output(out)

In [ ]:
# create a PatchSpec out of the Patch - we will use this to create a Synth
am_patch_spec = SimpleAMPatch().to_spec()
am_patch_spec

### One-off version

In [ ]:
# create a dict for params (for UI sliders)
am_params = {
    "carrier_freq": {
        "min": 20,
        "max": 8000,
        "unit": "Hz",
        "scale": "log"
    },
    "mod_freq": {
        "min": 0.1,
        "max": 100,
        "unit": "Hz",
        "scale": "log"
    },
    "mod_depth": {
        "min": 0,
        "max": 1,
    }
}

In [ ]:
simple_am_synth = Synth(am_patch_spec, am_params, name="SimpleAMSynth", add_amplitude=True, add_panning=True)
app_9.attach(simple_am_synth)
simple_am_synth.ui

### Long-term version

In [ ]:
class SimpleAM(Synth):
    def __init__(
            self,
            carrier_frequency=440,
            modulator_frequency=1,
            modulator_depth=0.25,
            name="SimpleAM"
    ):
        _spec = SimpleAMPatch(
                carr_freq=carrier_frequency,
                mod_freq=modulator_frequency,
                mod_depth=modulator_depth
        ).to_spec()
        _params = {
            "carrier_freq": {
                "min": 20,
                "max": 8000,
                "unit": "Hz",
                "scale": "log"
            },
            "mod_freq": {
                "min": 0.1,
                "max": 100,
                "unit": "Hz",
                "scale": "log"
            },
            "mod_depth": {
                "min": 0,
                "max": 1,
            }
        }
        # call the parent constructor
        super().__init__(_spec, params_dict=_params, name=name, add_amplitude=True, add_panning=True)

def __repr__(self): # not really necessary, but nice to have
    return f"SimpleAM {self.id}: {self.name}"

In [ ]:
simple_am = SimpleAM()
app_9.attach(simple_am)
simple_am.ui

## Multi-target mapping and custom Mappers

In [ ]:
from pixasonics.synths import Theremin, FilteredNoise
from pixasonics.features import MeanChannelValue

### Multi-target mapping

In [ ]:
app_10 = App()
app_10.load_image_file("images/test.jpg")
mean_red_10 = MeanChannelValue(filter_channels=0, name="MeanRed")
app_10.attach(mean_red_10)
theremin_10 = Theremin()
app_10.attach(theremin_10)
filtered_noise_10 = FilteredNoise()
app_10.attach(filtered_noise_10)
red2freqs_10 = Mapper(mean_red_10, [theremin_10["frequency"], filtered_noise_10["cutoff"]], name="Red2Freqs") # a list of targets!
app_10.attach(red2freqs_10)

In [ ]:
red2freqs_10.in_low, red2freqs_10.in_high, red2freqs_10.out_low, red2freqs_10.out_high, red2freqs_10.exponent

In [ ]:
# it is possible to set the ranges and exponent for each target separately
red2freqs_10.out_low = [60, 50]
red2freqs_10.out_high = [4000, 5000]
red2freqs_10.exponent = [1, 2]

red2freqs_10.out_low, red2freqs_10.out_high, red2freqs_10.exponent

### Custom Mappers

In [ ]:
app_11 = App()
app_11.load_image_file("images/test.jpg")
mean_pix_11 = MeanChannelValue()
app_11.attach(mean_pix_11)
simple_am_11 = SimpleAM() # from the custom synth example above
app_11.attach(simple_am_11)

In [ ]:
class RGB2AM(Mapper):
    def __init__(self, source, target_am, name="RGB2AM"):
        # here we already define a multitarget mapping, expecting the SimpleAM parameters
        super().__init__(source, [target_am["carrier_freq"], target_am["mod_freq"], target_am["mod_depth"]], name=name)
    
    def map(self, in_data):
        r, g, b = in_data # this is going to be the current values from a MeanChannelValue on an RGB image
        # map red to carrier frequency
        carr_freq = np.interp(r, [0, 255], [60, 1000])
        # map green to modulator frequency
        mod_freq = np.interp(g, [0, 255], [0.1, 100])
        # map blue to modulator depth
        mod_depth = np.interp(b, [0, 2], [0, 1])
        # return them as a list (as expected by the multitarget mapping we set up above)
        return [carr_freq, mod_freq, mod_depth]

In [ ]:
rgb2am = RGB2AM(mean_pix_11, simple_am_11)
app_11.attach(rgb2am)

## Multiple Apps and the AppRegistry

In [ ]:
from pixasonics.core import AppRegistry
AppRegistry._apps # list of all currently running Apps